# KabyLLM v1 · a Kabyle chat model trained from scratch

This notebook trains a small GPT-style language model **from scratch** on a private chat corpus written in romanized Kabyle
(with the French words, Arabic loanwords and emojis that come with real chats), then lets you talk to it.

**What happens when you run all cells**
1. The discussion is cleaned: "message deleted" notices, edit tags, links, e-mail addresses and phone numbers are removed.
2. A byte-level BPE tokenizer is trained on the discussion.
3. A Transformer with ~11M parameters is trained. At 0%, 10%, 20% … 100% of training it answers
   **"amek achu xedmed akka"**, so you can watch it go from random noise to Kabyle (11 answers).
4. The weights with the best validation loss are saved to `/kaggle/working/kabyle_gpt.pt`.
5. You can ask the model questions, or chat with it.

**Running on Kaggle**
- *Settings → Accelerator → GPU T4 ×1* (with *T4 ×2* only the first GPU is used; this model is too small to gain from two).
- *Add Input → Upload* `discussion.txt` as a dataset. The notebook finds it anywhere under `/kaggle/input`.
- Training takes about **2 minutes** on a T4 (first run: 3,000 steps in 4 min 47 s), far below the 12 h limit. What limits this model is the amount of text, not the GPU: training longer on the same discussion leads to memorization (see the table).

### Design choices

| | Choice | Why |
|---|---|---|
| **Tokenizer** | Byte-level BPE, 2,048 tokens, trained on the discussion | Every character (emoji, é, Arabic script) can be encoded, so there is no `<UNK>`. Digits stay inside words: `Ma3lich` is one token, not `Ma` `3` `lich` as with GPT-2's tokenizer. |
| **Messages** | A special end-of-message token after every message | The model learns where a message ends, and a reply is whatever it writes up to that token. The discussion keeps its original order, so the model also learns how messages follow each other, whoever wrote them. |
| **Architecture** | Decoder-only Transformer with pre-norm RMSNorm, rotary position embeddings (RoPE), SwiGLU feed-forward, tied input/output embeddings, no biases | The building blocks of Llama and Mistral, at a small scale. |
| **Size** | 6 layers · 384 dimensions · 6 heads · 256-token context (≈ 25 messages) · **11.4M parameters** | The discussion is about 570K training tokens. v0 had 43M parameters for ~1M tokens and memorized its data (train loss 1.33, val loss 6.21). |
| **Regularization** | Dropout 0.3, weight decay 0.1, keep the checkpoint with the best **validation** loss | With this little text, the main risk is the model memorizing the discussion instead of learning the language. |
| **Validation** | 10% of the discussion, as blocks of 200 consecutive messages spread over the whole timeline | Splitting line by line would leak, because neighbouring messages are often near duplicates. |
| **Precision** | float16 mixed precision with loss scaling | The T4 (Turing architecture) has no bfloat16 support. |

## 1 · Setup

In [ ]:
import contextlib, glob, math, os, random, re, textwrap, time
from collections import Counter
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

SEED = 1337
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    # Turing GPUs (T4) have no bfloat16 → float16 with loss scaling. Ampere and newer → bfloat16.
    AMP_DTYPE = torch.bfloat16 if gpu.major >= 8 else torch.float16
    print(f"GPU       : {gpu.name} · {gpu.total_memory / 1024**3:.1f} GB · compute capability {gpu.major}.{gpu.minor}")
else:
    AMP_DTYPE = None
    print("GPU       : none found, training on CPU will be very slow (Kaggle: Settings → Accelerator → GPU T4)")
print(f"PyTorch   : {torch.__version__}")
print(f"Precision : {'mixed ' + str(AMP_DTYPE).removeprefix('torch.') if AMP_DTYPE else 'float32'}")


def amp_context():
    """Mixed-precision autocast on GPU, no-op on CPU."""
    return torch.autocast("cuda", dtype=AMP_DTYPE) if AMP_DTYPE else contextlib.nullcontext()


def fmt_time(seconds):
    minutes, seconds = divmod(int(seconds), 60)
    hours, minutes = divmod(minutes, 60)
    return f"{hours}h {minutes:02d}m" if hours else f"{minutes}m {seconds:02d}s"

## 2 · Configuration

Every setting lives here. The defaults are sized for this discussion on a T4.

In [ ]:
# ── Data ─────────────────────────────────────────────────────────────────────
DATA_FILE        = None      # None → look for discussion.txt under /kaggle/input, then under the current folder
OUTPUT_DIR       = "/kaggle/working" if os.path.isdir("/kaggle/working") else "outputs"
MAX_CHAR_REPEAT  = 4         # "Sahiiiiiiiit" → "Sahiiiit", 18 × 🤣 → 4 × 🤣   (None keeps text as is)
VAL_FRACTION     = 0.10      # share of the discussion held out to measure generalization
VAL_BLOCK_MSGS   = 200       # validation is made of whole blocks of this many consecutive messages

# ── Tokenizer ────────────────────────────────────────────────────────────────
VOCAB_SIZE       = 2048

# ── Model (≈ 11.4M parameters) ───────────────────────────────────────────────
CONTEXT_LEN      = 256       # tokens seen at once (≈ 25 messages)
D_MODEL          = 384
N_LAYERS         = 6
N_HEADS          = 6
FFN_HIDDEN       = 1024      # SwiGLU hidden size, ≈ 8/3 × D_MODEL
DROPOUT          = 0.3       # run 1 used 0.2

# ── Training ─────────────────────────────────────────────────────────────────
MAX_ITERS        = 1200      # run 1: 3,000 steps → best val loss 4.04 at step 750, memorization after that
BATCH_SIZE       = 32        # sequences per step → 32 × 256 = 8,192 tokens per step
LEARNING_RATE    = 1e-3      # peak, after warmup
MIN_LR           = 1e-4      # end of the cosine decay
WARMUP_ITERS     = 100
WEIGHT_DECAY     = 0.1
GRAD_CLIP        = 1.0
N_EVALS          = 40        # silent validation checks (shown in the progress bar, used to keep the best weights)

# ── Progress snapshots and chat ──────────────────────────────────────────────
PROBE_PROMPT     = "amek achu xedmed akka"
N_SNAPSHOTS      = 11        # answers at 0%, 10%, …, 100% of training
REPLY_MESSAGES   = 3         # a reply = the next N messages the model writes
MAX_REPLY_TOKENS = 120       # hard stop if the model never ends its messages
TEMPERATURE      = 0.8       # < 1 = safer, > 1 = more creative
TOP_K            = 50
TOP_P            = 0.95
SNAPSHOT_SEED    = 42        # same random seed at every snapshot, so answers differ because of training, not luck

os.makedirs(OUTPUT_DIR, exist_ok=True)
CHECKPOINT_PATH = os.path.join(OUTPUT_DIR, "kabyle_gpt.pt")

## 3 · Load and clean the discussion

One line = one message. Messages stay in their original order.

In [ ]:
RE_NOTICE = re.compile(r"^(?:Vous avez supprimé ce message|Ce message a été supprimé)\.?$", re.IGNORECASE)
RE_TAG    = re.compile(r"<[^<>]*(?:modifié|omis|supprimé)[^<>]*>", re.IGNORECASE)
RE_URL    = re.compile(r"(?:https?://|\bwww\.)\S+", re.IGNORECASE)
RE_EMAIL  = re.compile(r"\S+@\S+\.[a-z]{2,}", re.IGNORECASE)
RE_PHONE  = re.compile(r"(?:\+|\b00)\d{3}[\s.-]?\d(?:[\s.-]?\d{2}){4}\b|\b0[5-7]\d{2}(?:[\s.-]?\d{2}){3}\b")
RE_REPEAT = re.compile(r"(.{1,2}?)\1{%d,}" % MAX_CHAR_REPEAT) if MAX_CHAR_REPEAT else None


def find_data_file():
    if DATA_FILE:
        return DATA_FILE
    for pattern in ("/kaggle/input/**/discussion.txt", "**/discussion.txt"):
        hits = sorted(glob.glob(pattern, recursive=True))
        if hits:
            return hits[0]
    raise FileNotFoundError("discussion.txt not found. Add it as a Kaggle dataset, or set DATA_FILE in the configuration cell.")


def clean_message(line, stats):
    """Return the cleaned message, or None if nothing worth keeping is left."""
    line = line.strip()
    if not line:
        stats["empty lines"] += 1
        return None
    if RE_NOTICE.match(line):
        stats["'message deleted' notices"] += 1
        return None
    for label, pattern in (("edit / voice-note tags", RE_TAG), ("links", RE_URL),
                           ("e-mail addresses", RE_EMAIL), ("phone numbers", RE_PHONE)):
        line, n = pattern.subn("", line)
        stats[label] += n
    if RE_REPEAT:
        line, n = RE_REPEAT.subn(lambda m: m.group(1) * MAX_CHAR_REPEAT, line)
        stats["long repetitions shortened"] += n
    line = re.sub(r"\s+", " ", line).strip()
    if not line:
        stats["lines left empty after cleaning"] += 1
        return None
    return line


data_path = find_data_file()
with open(data_path, encoding="utf-8") as f:
    raw_lines = f.read().splitlines()

cleaning_stats = Counter()
messages = [m for m in (clean_message(line, cleaning_stats) for line in raw_lines) if m is not None]

print(f"File       : {data_path}")
print(f"Messages   : {len(raw_lines):,} lines → {len(messages):,} messages kept")
print(f"Characters : {sum(map(len, raw_lines)):,} → {sum(map(len, messages)):,}")
print("\nCleaning")
for label in ("empty lines", "'message deleted' notices", "edit / voice-note tags", "links",
              "e-mail addresses", "phone numbers", "long repetitions shortened", "lines left empty after cleaning"):
    if label in cleaning_stats:
        print(f"  {label:.<36} {cleaning_stats[label]:>6,}")
print("\nRandom sample")
for m in random.Random(SEED).sample(messages, 8):
    print(f"  │ {m}")

## 4 · Tokenizer: byte-level BPE

BPE starts from the 256 possible bytes and repeatedly merges the most frequent adjacent pair
(`a`+`k` → `ak`, `ak`+`ka` → `akka`, …) until the vocabulary reaches `VOCAB_SIZE`.
Frequent words become single tokens, and rare words are spelled out with smaller pieces.

In [ ]:
from tokenizers import Tokenizer, Regex, decoders, models, pre_tokenizers, trainers

MSG_TOKEN = "<|msg|>"
# Text is first cut into words, then BPE works inside each word. Unlike GPT-2's rule, digits stay attached
# to letters, because in romanized Kabyle they are letters (3 = ɛ, as in "Ma3lich").
WORD_SPLIT = r" ?[\p{L}\p{N}][\p{L}\p{N}\p{M}]*| ?[^\s\p{L}\p{N}]+|\s+"

tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Split(Regex(WORD_SPLIT), behavior="isolated"),
    pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
])
tokenizer.decoder = decoders.ByteLevel()
tokenizer.train_from_iterator(messages, trainers.BpeTrainer(
    vocab_size=VOCAB_SIZE, min_frequency=2, special_tokens=[MSG_TOKEN],
    initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False,
))
MSG_ID = tokenizer.token_to_id(MSG_TOKEN)


def show_tokens(text):
    return " ".join(f"[{tokenizer.decode([i])}]" for i in tokenizer.encode(text).ids)


message_ids = [e.ids for e in tokenizer.encode_batch(messages)]
n_tokens = sum(map(len, message_ids)) + len(message_ids)  # + one <|msg|> per message
n_roundtrip_ok = sum(tokenizer.decode(ids) == m for ids, m in zip(message_ids, messages))

print(f"Vocabulary      : {tokenizer.get_vocab_size():,} tokens (256 bytes + merges + {MSG_TOKEN})")
print(f"Corpus          : {n_tokens:,} tokens · {sum(map(len, messages)) / n_tokens:.2f} characters per token")
print(f"Lossless decode : {n_roundtrip_ok:,} / {len(messages):,} messages")
print("\nExamples")
for text in (PROBE_PROMPT, "Ma3lich 😂😂😂", "Bonne nuit beaux rêves"):
    print(f"  {text!r:<28} → {show_tokens(text)}")

## 5 · Training data

All messages are joined into one long token stream, `<|msg|>` after each message.
A training example is a random window of `CONTEXT_LEN` tokens, and the model learns to predict each next token.

In [ ]:
n_blocks = math.ceil(len(message_ids) / VAL_BLOCK_MSGS)
val_blocks = set(random.Random(SEED).sample(range(n_blocks), max(1, round(n_blocks * VAL_FRACTION))))


def build_stream(blocks):
    stream, n_msgs = [MSG_ID], 0
    for b in sorted(blocks):
        for ids in message_ids[b * VAL_BLOCK_MSGS:(b + 1) * VAL_BLOCK_MSGS]:
            stream += ids + [MSG_ID]
            n_msgs += 1
    return torch.tensor(stream, dtype=torch.long, device=DEVICE), n_msgs


train_data, n_train_msgs = build_stream(set(range(n_blocks)) - val_blocks)
val_data, n_val_msgs = build_stream(val_blocks)


def get_batch():
    """BATCH_SIZE random windows from the training stream; targets are the inputs shifted by one token."""
    starts = torch.randint(len(train_data) - CONTEXT_LEN - 1, (BATCH_SIZE, 1), device=DEVICE)
    windows = train_data[starts + torch.arange(CONTEXT_LEN + 1, device=DEVICE)]
    return windows[:, :-1], windows[:, 1:]


def fixed_windows(data, starts):
    idx = starts[:, None] + torch.arange(CONTEXT_LEN + 1, device=DEVICE)
    windows = data[idx]
    return windows[:, :-1], windows[:, 1:]


# Evaluation always uses the same windows, so the losses are comparable from one check to the next:
# the whole validation stream cut into consecutive windows, and as many fixed random windows from the training stream.
n_val_windows = (len(val_data) - 1) // CONTEXT_LEN
eval_sets = {
    "val": fixed_windows(val_data, torch.arange(n_val_windows, device=DEVICE) * CONTEXT_LEN),
    "train": fixed_windows(train_data, torch.randint(len(train_data) - CONTEXT_LEN - 1, (n_val_windows,),
                                                     generator=torch.Generator().manual_seed(SEED)).to(DEVICE)),
}

tokens_per_step = BATCH_SIZE * CONTEXT_LEN
print(f"Train      : {len(train_data):>9,} tokens · {n_train_msgs:,} messages")
print(f"Validation : {len(val_data):>9,} tokens · {n_val_msgs:,} messages ({len(val_blocks)} blocks spread over the discussion)")
print(f"One step   : {BATCH_SIZE} × {CONTEXT_LEN} = {tokens_per_step:,} tokens")
print(f"Training   : {MAX_ITERS:,} steps = {MAX_ITERS * tokens_per_step / 1e6:.1f}M tokens ≈ "
      f"{MAX_ITERS * tokens_per_step / len(train_data):.0f} passes over the training data")

## 6 · Model architecture

```
tokens → embedding → [ RMSNorm → causal self-attention (RoPE) → + ]  × N_LAYERS → RMSNorm → linear → next-token scores
                     [ RMSNorm → SwiGLU feed-forward           → + ]
```
- **RMSNorm** rescales vectors to a fixed size before each sub-layer ("pre-norm"), which keeps training stable.
- **Causal self-attention** lets each token look at earlier tokens only. **RoPE** encodes positions by rotating queries and keys, so attention depends on the distance between tokens.
- **SwiGLU** is a gated feed-forward layer. It trains better than the classic GELU MLP.
- **Weight tying**: the output layer reuses the embedding matrix, which saves parameters and helps on small data.

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int
    context_len: int = 256
    d_model: int = 384
    n_layers: int = 6
    n_heads: int = 6
    ffn_hidden: int = 1024
    dropout: float = 0.2
    rope_theta: float = 10_000.0


class RMSNorm(nn.Module):
    """Divide each vector by its root-mean-square, then apply a learned per-channel gain."""

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x_float = x.float()  # computed in float32 for stability under fp16
        normed = x_float * torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return normed.type_as(x) * self.weight


def rope_tables(cfg):
    """cos / sin of the rotation angles for every (position, frequency): shape (context_len, head_dim / 2)."""
    head_dim = cfg.d_model // cfg.n_heads
    inv_freq = 1.0 / (cfg.rope_theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    angles = torch.outer(torch.arange(cfg.context_len).float(), inv_freq)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    """Rotate channel pairs of x (batch, heads, time, head_dim) by position-dependent angles."""
    x1, x2 = x.chunk(2, dim=-1)
    cos, sin = cos.to(x.dtype), sin.to(x.dtype)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1)


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        assert cfg.d_model % cfg.n_heads == 0, "D_MODEL must be divisible by N_HEADS"
        self.n_heads = cfg.n_heads
        self.dropout = cfg.dropout
        self.w_qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=False)
        self.w_out = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.out_dropout = nn.Dropout(cfg.dropout)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.w_qkv(x).split(C, dim=-1)
        q, k, v = (t.view(B, T, self.n_heads, C // self.n_heads).transpose(1, 2) for t in (q, k, v))
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        # Fused attention kernel; is_causal=True hides future tokens.
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_dropout(self.w_out(y))


class SwiGLU(nn.Module):
    """Gated feed-forward layer: w_down( SiLU(w_gate(x)) * w_up(x) )."""

    def __init__(self, cfg):
        super().__init__()
        self.w_gate = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_up = nn.Linear(cfg.d_model, cfg.ffn_hidden, bias=False)
        self.w_down = nn.Linear(cfg.ffn_hidden, cfg.d_model, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        return self.dropout(self.w_down(F.silu(self.w_gate(x)) * self.w_up(x)))


class Block(nn.Module):
    """One Transformer layer: x + attention(norm(x)), then x + feed_forward(norm(x))."""

    def __init__(self, cfg):
        super().__init__()
        self.attn_norm = RMSNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ffn_norm = RMSNorm(cfg.d_model)
        self.ffn = SwiGLU(cfg)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.attn_norm(x), cos, sin)
        return x + self.ffn(self.ffn_norm(x))


class KabyleGPT(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.emb_dropout = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # weight tying

        cos, sin = rope_tables(cfg)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

        self.apply(self._init_weights)
        # GPT-2 initialization: layers writing into the residual stream start smaller, so its scale does not grow with depth.
        for name, p in self.named_parameters():
            if name.endswith(("w_out.weight", "w_down.weight")):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layers))

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        T = idx.size(1)
        assert T <= self.cfg.context_len, f"{T} tokens exceed the context length ({self.cfg.context_len})"
        x = self.emb_dropout(self.tok_emb(idx))
        cos, sin = self.rope_cos[:T], self.rope_sin[:T]
        for block in self.blocks:
            x = block(x, cos, sin)
        logits = self.lm_head(self.final_norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

In [ ]:
model_cfg = ModelConfig(vocab_size=tokenizer.get_vocab_size(), context_len=CONTEXT_LEN, d_model=D_MODEL,
                        n_layers=N_LAYERS, n_heads=N_HEADS, ffn_hidden=FFN_HIDDEN, dropout=DROPOUT)
model = KabyleGPT(model_cfg).to(DEVICE)

param_counts = dict.fromkeys(["Token embeddings (shared with output)", "Attention", "Feed-forward (SwiGLU)", "RMSNorm gains"], 0)
for name, p in model.named_parameters():  # the tied output layer is listed once, under tok_emb
    param_counts["Token embeddings (shared with output)" if name.startswith("tok_emb") else
                 "Attention" if ".attn." in name else
                 "Feed-forward (SwiGLU)" if ".ffn." in name else
                 "RMSNorm gains"] += p.numel()
n_params = sum(param_counts.values())

print(f"KabyleGPT · {N_LAYERS} layers · d_model {D_MODEL} · {N_HEADS} heads of {D_MODEL // N_HEADS} dims · "
      f"SwiGLU {FFN_HIDDEN} · context {CONTEXT_LEN} tokens · vocabulary {model_cfg.vocab_size:,}\n")
for label, count in param_counts.items():
    print(f"  {label:<40}{count:>12,}")
print(f"  {'─' * 52}\n  {'Total':<40}{n_params:>12,}  ({n_params / 1e6:.1f}M)")

## 7 · Text generation

The model writes one token at a time. At each step it scores every token in the vocabulary, and one token is drawn at random among the most likely:
- **temperature** sharpens (< 1) or flattens (> 1) the distribution,
- **top-k** keeps only the k best tokens, **top-p** keeps the smallest set of tokens covering p of the probability.

The conversation so far is fed as `<|msg|> message <|msg|> message <|msg|>`, and generation stops after `REPLY_MESSAGES` messages.

In [ ]:
def encode(text):
    return tokenizer.encode(text).ids


@torch.no_grad()
def generate_messages(model, conversation, n_messages=REPLY_MESSAGES, max_new_tokens=MAX_REPLY_TOKENS,
                      temperature=TEMPERATURE, top_k=TOP_K, top_p=TOP_P, seed=None):
    """Continue a conversation (list of message strings) and return the next n_messages messages."""
    was_training = model.training
    model.eval()
    ids = [MSG_ID]
    for message in conversation:
        ids += encode(message) + [MSG_ID]
    idx = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    generator = torch.Generator(device=DEVICE).manual_seed(seed) if seed is not None else None

    replies, current = [], []
    for _ in range(max_new_tokens):
        with amp_context():
            logits, _ = model(idx[:, -model.cfg.context_len:])
        logits = logits[0, -1].float() / max(temperature, 1e-5)
        if top_k:
            kth_best = torch.topk(logits, min(top_k, logits.numel())).values[-1]
            logits[logits < kth_best] = float("-inf")
        if top_p < 1.0:
            sorted_logits, order = torch.sort(logits, descending=True)
            sorted_probs = sorted_logits.softmax(-1)
            outside = sorted_probs.cumsum(-1) - sorted_probs > top_p  # the most likely token is always kept
            logits[order[outside]] = float("-inf")
        next_id = torch.multinomial(logits.softmax(-1), 1, generator=generator)
        idx = torch.cat([idx, next_id.view(1, 1)], dim=1)

        token = next_id.item()
        if token != MSG_ID:
            current.append(token)
            continue
        text = tokenizer.decode(current).strip()
        current = []
        if text:
            replies.append(text)
            if len(replies) == n_messages:
                break
    if current:  # ran out of tokens in the middle of a message
        replies.append(tokenizer.decode(current).strip() + " …")
    model.train(was_training)
    return replies


def format_exchange(prompt, replies, indent="    ", width=110):
    lines = [f"{indent}you   › {prompt}"]
    for i, reply in enumerate(replies or ["(no answer)"]):
        prefix = indent + ("model › " if i == 0 else "        ")
        lines.append(textwrap.fill(reply, width=width, initial_indent=prefix,
                                   subsequent_indent=" " * len(prefix)) or prefix)
    return "\n".join(lines)

## 8 · Training

- **AdamW** optimizer, with weight decay only on weight matrices (not on RMSNorm gains).
- **Learning rate**: linear warmup for `WARMUP_ITERS` steps, then cosine decay from `LEARNING_RATE` to `MIN_LR`.
- **Gradient clipping** at norm 1.0, and **float16 mixed precision** with a gradient scaler.
- Train and validation losses are measured `N_EVALS` times without printing anything. When the validation loss is the best so far, the weights are saved.
- At every 10% of training, the model answers `PROBE_PROMPT`.

The loss is the cross-entropy on the next token. A model guessing uniformly at random scores ln(2048) ≈ 7.62.
**If validation loss rises while training loss keeps falling, the model is memorizing the discussion.** The saved checkpoint is the one from before that point.

In [ ]:
decay = [p for p in model.parameters() if p.dim() >= 2]
no_decay = [p for p in model.parameters() if p.dim() < 2]
optimizer = torch.optim.AdamW(
    [{"params": decay, "weight_decay": WEIGHT_DECAY}, {"params": no_decay, "weight_decay": 0.0}],
    lr=LEARNING_RATE, betas=(0.9, 0.95), **({"fused": True} if DEVICE == "cuda" else {}),
)
scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)


def lr_at(step):
    if step < WARMUP_ITERS:
        return LEARNING_RATE * (step + 1) / WARMUP_ITERS
    progress = min(1.0, (step - WARMUP_ITERS) / max(1, MAX_ITERS - WARMUP_ITERS))
    return MIN_LR + 0.5 * (LEARNING_RATE - MIN_LR) * (1 + math.cos(math.pi * progress))


@torch.no_grad()
def evaluate(batch_size=64):
    model.eval()
    losses = {}
    for split, (xs, ys) in eval_sets.items():
        total = 0.0
        for i in range(0, len(xs), batch_size):
            with amp_context():
                _, loss = model(xs[i:i + batch_size], ys[i:i + batch_size])
            total += loss.item() * len(xs[i:i + batch_size])
        losses[split] = total / len(xs)
    model.train()
    return losses


def save_checkpoint(path, **extra):
    torch.save({
        "model": model.state_dict(),
        "model_config": asdict(model_cfg),
        "tokenizer": tokenizer.to_str(),
        "msg_token": MSG_TOKEN,
        "sampling": {"temperature": TEMPERATURE, "top_k": TOP_K, "top_p": TOP_P, "reply_messages": REPLY_MESSAGES},
        **extra,
    }, path)


def snapshot_report(number, step, losses, replies, elapsed):
    title = f" Snapshot {number:>2}/{N_SNAPSHOTS} · {round(100 * step / MAX_ITERS):>3}% · step {step:,}/{MAX_ITERS:,} "
    return "\n".join([
        "",
        "━━━" + title + "━" * max(0, 80 - len(title)),
        f"    train loss {losses['train']:.3f} · val loss {losses['val']:.3f} · elapsed {fmt_time(elapsed)}",
        format_exchange(PROBE_PROMPT, replies),
    ])


def progress_postfix(train_loss, lr):
    return f"loss {train_loss:.3f} · val {history['val'][-1]:.3f} · best val {best['val']:.3f} · lr {lr:.1e}"


snapshot_steps = [round(i * MAX_ITERS / (N_SNAPSHOTS - 1)) for i in range(N_SNAPSHOTS)]
eval_steps = set(snapshot_steps) | set(range(0, MAX_ITERS + 1, max(1, MAX_ITERS // N_EVALS)))
history = {"step": [], "train": [], "val": []}
best = {"val": float("inf"), "step": 0}

print(f"Training KabyleGPT ({n_params / 1e6:.1f}M parameters) for {MAX_ITERS:,} steps on {DEVICE.upper()}.")
print(f"It answers {PROBE_PROMPT!r} at 0%, 10%, …, 100%. A random guess scores a loss of {math.log(model_cfg.vocab_size):.2f}.")

model.train()
start = time.time()
smooth_loss, lr = None, lr_at(0)
progress = tqdm(total=MAX_ITERS, desc="Training", unit="step", dynamic_ncols=True, smoothing=0.05)
for step in range(MAX_ITERS + 1):
    if step in eval_steps:
        losses = evaluate()
        for key, value in (("step", step), ("train", losses["train"]), ("val", losses["val"])):
            history[key].append(value)
        if losses["val"] < best["val"]:
            best = {"val": losses["val"], "step": step}
            save_checkpoint(CHECKPOINT_PATH, step=step, train_loss=losses["train"], val_loss=losses["val"])
        if smooth_loss is not None:
            progress.set_postfix_str(progress_postfix(smooth_loss, lr))

    if step in snapshot_steps:
        replies = generate_messages(model, [PROBE_PROMPT], seed=SNAPSHOT_SEED)
        tqdm.write(snapshot_report(snapshot_steps.index(step) + 1, step, losses, replies, time.time() - start))

    if step == MAX_ITERS:
        break

    lr = lr_at(step)
    for group in optimizer.param_groups:
        group["lr"] = lr
    x, y = get_batch()
    with amp_context():
        _, loss = model(x, y)
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()

    progress.update(1)
    if step % 10 == 0:  # reading the loss forces a GPU sync, so not at every step
        smooth_loss = loss.item() if smooth_loss is None else 0.8 * smooth_loss + 0.2 * loss.item()
        progress.set_postfix_str(progress_postfix(smooth_loss, lr), refresh=False)
progress.close()

train_time = time.time() - start
print(f"\nDone in {fmt_time(train_time)} · {MAX_ITERS * tokens_per_step / train_time:,.0f} tokens/s (evaluations included)")
print(f"Best validation loss {best['val']:.3f} at step {best['step']:,} ({100 * best['step'] / MAX_ITERS:.0f}% of training) → {CHECKPOINT_PATH}")
val_at_80 = next(v for s, v in zip(history["step"], history["val"]) if s >= 0.8 * MAX_ITERS)
if best["step"] < 0.7 * MAX_ITERS:
    print("Validation loss stopped improving before the end: after that point the model was memorizing the discussion.\n"
          "The saved model is the best one. For a next run, lower MAX_ITERS or raise DROPOUT.")
elif best["step"] == MAX_ITERS and val_at_80 - best["val"] > 0.02:
    print("Validation loss was still going down at the end: a longer run (higher MAX_ITERS) may give a better model.")

## 9 · Training curves

In [ ]:
import matplotlib.pyplot as plt

TRAIN_COLOR, VAL_COLOR = "#2a78d6", "#eb6834"
INK, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#e6e5e0", "#c9c8c2"

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(history["step"], history["train"], color=TRAIN_COLOR, lw=2, label="train")
ax.plot(history["step"], history["val"], color=VAL_COLOR, lw=2, label="validation")
ax.plot(best["step"], best["val"], "o", ms=8, color=VAL_COLOR, mec="white", mew=2, zorder=3)
ax.annotate(f"best val {best['val']:.3f} · step {best['step']:,}", (best["step"], best["val"]),
            xytext=(0, 10), textcoords="offset points", ha="center", va="bottom", color=MUTED)

# Direct labels at the end of each line, pushed apart when the two lines end close together.
y_min, y_max = min(history["train"] + history["val"]), max(history["train"] + history["val"])
y_train, y_val = history["train"][-1], history["val"][-1]
min_gap = 0.06 * (y_max - y_min)
if abs(y_val - y_train) < min_gap:
    middle, sign = (y_val + y_train) / 2, (1 if y_val >= y_train else -1)
    y_train, y_val = middle - sign * min_gap / 2, middle + sign * min_gap / 2
for y, text in ((y_train, "train"), (y_val, "validation")):
    ax.annotate(text, (history["step"][-1], y), xytext=(8, 0), textcoords="offset points", va="center", color=MUTED)
ax.set_xlim(0, MAX_ITERS * 1.12)

ax.set_title("Training and validation loss", loc="left", color=INK, fontsize=13)
ax.set_xlabel("step", color=MUTED)
ax.set_ylabel("cross-entropy loss", color=MUTED)
ax.grid(axis="y", color=GRID, lw=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color(AXIS)
ax.tick_params(colors=MUTED)
ax.legend(frameon=False, loc="upper right", labelcolor=MUTED)
plt.tight_layout()
plt.show()

## 10 · Save the model

The best weights were already written during training. This cell loads them back into `model` (so the chat below uses them)
and adds the training history to the file.

`kabyle_gpt.pt` contains everything needed to use the model again: weights, architecture and tokenizer.
To download it on Kaggle: *Output* panel on the right → `/kaggle/working` → `kabyle_gpt.pt` → ⋮ → Download.

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True)
model.load_state_dict(checkpoint["model"])
model.eval()
save_checkpoint(CHECKPOINT_PATH, step=checkpoint["step"], train_loss=checkpoint["train_loss"],
                val_loss=checkpoint["val_loss"], history=history, n_params=n_params, train_time_s=train_time)

print(f"Loaded the best weights (step {checkpoint['step']:,}, val loss {checkpoint['val_loss']:.3f}) into `model`.")
print(f"Saved {CHECKPOINT_PATH} ({os.path.getsize(CHECKPOINT_PATH) / 1024**2:.1f} MB)")
try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(CHECKPOINT_PATH), result_html_prefix="Download: "))
except ImportError:
    pass

## 11 · Talk to the model

`ask("…")` sends one message and prints the answer. Change `temperature`, `top_k`, `top_p` or `n_messages` to see how the answers change.

In [ ]:
def ask(message, **options):
    print(format_exchange(message, generate_messages(model, [message], **options)), end="\n\n")


ask("amek achu xedmed akka")
ask("Azul, amek telid ?")
ask("Achu txedmed ass agi ?")
ask("Bonne nuit")
ask("amek achu xedmed akka", temperature=1.1, n_messages=5)

`chat()` starts a conversation that remembers what was said, within the 256-token context.
Type `reset` to start over and `quit` to stop. It needs an interactive session: in a *Save & Run All* run, use `ask()` instead.

In [ ]:
def chat(n_messages=REPLY_MESSAGES, **options):
    print("Chat with KabyleGPT · 'reset' starts over · 'quit' stops\n")
    conversation = []
    while True:
        try:
            message = input("you › ").strip()
        except (NotImplementedError, EOFError):
            print("(No keyboard input in this session. Use ask(\"...\") instead.)")
            return
        except KeyboardInterrupt:
            print("Bye!")
            return
        if message.lower() in {"quit", "exit", "q"}:
            print("Bye!")
            return
        if message.lower() == "reset":
            conversation = []
            print("(new conversation)\n")
            continue
        if not message:
            continue
        conversation.append(message)
        replies = generate_messages(model, conversation, n_messages=n_messages, **options)
        conversation += [r.removesuffix(" …") for r in replies]
        for reply in replies or ["(no answer)"]:
            print(f"model › {reply}")
        print()


chat()

## 12 · Use the saved model later

In a new session (on Kaggle, add `kabyle_gpt.pt` as a dataset): run **1 · Setup**, **2 · Configuration**, **6 · Model architecture** (first code cell)
and **7 · Text generation**, then the cell below with the path to your file.

In [ ]:
from tokenizers import Tokenizer


def load_kabyle_gpt(path):
    """Return (model, tokenizer, id of the <|msg|> token) from a saved kabyle_gpt.pt."""
    ckpt = torch.load(path, map_location=DEVICE, weights_only=True)
    loaded_tokenizer = Tokenizer.from_str(ckpt["tokenizer"])
    loaded_model = KabyleGPT(ModelConfig(**ckpt["model_config"])).to(DEVICE)
    loaded_model.load_state_dict(ckpt["model"])
    return loaded_model.eval(), loaded_tokenizer, loaded_tokenizer.token_to_id(ckpt["msg_token"])


# Uncomment, set the path, run (then use ask("...") in the chat cell of section 11):
# model, tokenizer, MSG_ID = load_kabyle_gpt("/kaggle/input/<your-dataset>/kabyle_gpt.pt")
# ask("amek achu xedmed akka")